### this is a good next QC step after the automated QC, we'll start at the longest duration (72 hours), and then remove the obviously bad points at these timescales all the way down to 24 hours. we'll look visually at the 1000-yr, 72-hr maps to identify the obviously bad points at these timescales (which should be some of the most egregious)

### do this after running the automated QC with "find_exceedances_qc_vs_prism"

In [3]:
import pandas as pd
import os

In [189]:
duration = 72
dataset = "prism" ### which dataset to analyze

year = 2001

workdir = "/glade/work/rschumac/precip_data/extremerain/"


#### read in the exceedance files into pandas dataframes

In [190]:
df100_72h = pd.read_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_100y72h.csv")
df10_72h = pd.read_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_10y72h.csv")
try:
    df1000_72h = pd.read_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_1000y72h.csv")
except:  ## if no rows, just make an empty dataframe with the right columns
    df1000_72h = pd.DataFrame(columns=df100_72h.columns)

### and the 48-h ones
df100_48h = pd.read_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_100y48h.csv")
df10_48h = pd.read_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_10y48h.csv")
try:
    df1000_48h = pd.read_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_1000y48h.csv")
except:
    df1000_48h = pd.DataFrame(columns=df100_48h.columns)

### and the 24-h ones
df100_24h = pd.read_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_100y24h.csv")
df10_24h = pd.read_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_10y24h.csv")
try:
    df1000_24h = pd.read_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_all_points_1000y24h.csv")
except:
    df1000_24h = pd.DataFrame(columns=df100_24h.columns)


In [191]:
df1000_72h

,time,lat,lon,tp,tp_minus_ari,tp_pct_of_ari,event_num
0,2001-11-29 12:00:00,36.208333,-88.125000,362.27200,18.157002,1.052764,0
1,2001-11-29 12:00:00,36.208333,-88.083333,372.46698,27.757692,1.080525,0
2,2001-11-29 12:00:00,36.208333,-88.041667,350.77400,5.331252,1.015433,0
3,2001-11-29 12:00:00,36.250000,-88.125000,366.62900,22.511193,1.065417,0
4,2001-11-29 12:00:00,36.250000,-88.083333,378.26500,33.456505,1.097029,0
5,2001-11-29 12:00:00,36.250000,-88.041667,357.73700,12.135916,1.035115,0


### subset down to just events for each threshold, for looping

In [192]:
events1000_72h = df1000_72h.drop_duplicates(['time','event_num'])
events100_72h = df100_72h.drop_duplicates(['time','event_num'])
events10_72h = df10_72h.drop_duplicates(['time','event_num'])

print("1000-yr events: "+str(len(events1000_72h)))
print("100-yr events: "+str(len(events100_72h)))
print("10-yr events: "+str(len(events10_72h)))

events1000_72h

1000-yr events: 1
100-yr events: 22
10-yr events: 244


,time,lat,lon,tp,tp_minus_ari,tp_pct_of_ari,event_num
0,2001-11-29 12:00:00,36.208333,-88.125,362.272,18.157002,1.052764,0


## Pause here!

#### now, take a look at the maps (in this case, the 'compare' maps that were made elsewhere)

#### if we want to remove a row because it's bad, specify the row(s) in the 1000-year dataframe

In [151]:
df1000_72h[df1000_72h.time=='1995-04-20 12:00:00']

,time,lat,lon,tp,tp_minus_ari,tp_pct_of_ari,event_num


In [193]:
####           ### which index(es) to drop?
#to_drop = [0,1,2] + list(range(40,50))
           
           #### this needs to be user-specified based on manual inspection!!
#to_drop = list(range(0,83)) 
to_drop = []
####

rows_to_drop_visual = df1000_72h.loc[to_drop]
rows_to_drop_visual

,time,lat,lon,tp,tp_minus_ari,tp_pct_of_ari,event_num


#### find any other points within a small radius around these (say 0.25 degrees in each direction)
#### furthermore, find points in the 100- and 10-year files on the same day and in the same radius, in all 3 durations (72,48,24)

In [194]:
latlon_buf = 0.25  ## lat/lon buffer around the centerpoint

rows_to_drop_1000_72h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_100_72h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_10_72h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_1000_48h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_100_48h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_10_48h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_1000_24h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_100_24h = pd.DataFrame(columns=rows_to_drop_visual.columns)
rows_to_drop_10_24h = pd.DataFrame(columns=rows_to_drop_visual.columns)

for i in range(0,len(rows_to_drop_visual)):
    this_lon = rows_to_drop_visual.iloc[i].lon
    this_lat = rows_to_drop_visual.iloc[i].lat
    this_time = rows_to_drop_visual.iloc[i].time
    
    more_rows_to_drop_1000_72h = df1000_72h[(df1000_72h.time==this_time) & (df1000_72h.lon >= this_lon-latlon_buf) & (df1000_72h.lon <= this_lon+latlon_buf) & (df1000_72h.lat >= this_lat-latlon_buf) & (df1000_72h.lat <= this_lat+latlon_buf)] 
    rows_to_drop_1000_72h = pd.concat([rows_to_drop_1000_72h,more_rows_to_drop_1000_72h])

    ### now also find the rows in the same area in the 100 & 10-year files
    more_rows_to_drop_100_72h = df100_72h[(df100_72h.time==this_time) & (df100_72h.lon >= this_lon-latlon_buf) & (df100_72h.lon <= this_lon+latlon_buf) & (df100_72h.lat >= this_lat-latlon_buf) & (df100_72h.lat <= this_lat+latlon_buf)] 
    more_rows_to_drop_10_72h = df10_72h[(df10_72h.time==this_time) & (df10_72h.lon >= this_lon-latlon_buf) & (df10_72h.lon <= this_lon+latlon_buf) & (df10_72h.lat >= this_lat-latlon_buf) & (df10_72h.lat <= this_lat+latlon_buf)] 

    rows_to_drop_100_72h = pd.concat([rows_to_drop_100_72h,more_rows_to_drop_100_72h])
    rows_to_drop_10_72h = pd.concat([rows_to_drop_10_72h,more_rows_to_drop_10_72h])  

    ### repeat for 48h
    more_rows_to_drop_1000_48h = df1000_48h[(df1000_48h.time==this_time) & (df1000_48h.lon >= this_lon-latlon_buf) & (df1000_48h.lon <= this_lon+latlon_buf) & (df1000_48h.lat >= this_lat-latlon_buf) & (df1000_48h.lat <= this_lat+latlon_buf)] 
    rows_to_drop_1000_48h = pd.concat([rows_to_drop_1000_48h,more_rows_to_drop_1000_48h])
    more_rows_to_drop_100_48h = df100_48h[(df100_48h.time==this_time) & (df100_48h.lon >= this_lon-latlon_buf) & (df100_48h.lon <= this_lon+latlon_buf) & (df100_48h.lat >= this_lat-latlon_buf) & (df100_48h.lat <= this_lat+latlon_buf)] 
    more_rows_to_drop_10_48h = df10_48h[(df10_48h.time==this_time) & (df10_48h.lon >= this_lon-latlon_buf) & (df10_48h.lon <= this_lon+latlon_buf) & (df10_48h.lat >= this_lat-latlon_buf) & (df10_48h.lat <= this_lat+latlon_buf)] 
    rows_to_drop_100_48h = pd.concat([rows_to_drop_100_48h,more_rows_to_drop_100_48h])
    rows_to_drop_10_48h = pd.concat([rows_to_drop_10_48h,more_rows_to_drop_10_48h])  

    ### and 24
    more_rows_to_drop_1000_24h = df1000_24h[(df1000_24h.time==this_time) & (df1000_24h.lon >= this_lon-latlon_buf) & (df1000_24h.lon <= this_lon+latlon_buf) & (df1000_24h.lat >= this_lat-latlon_buf) & (df1000_24h.lat <= this_lat+latlon_buf)] 
    rows_to_drop_1000_24h = pd.concat([rows_to_drop_1000_24h,more_rows_to_drop_1000_24h])
    
    ### now also find the rows in the same area in the 100 & 10-year files
    more_rows_to_drop_100_24h = df100_24h[(df100_24h.time==this_time) & (df100_24h.lon >= this_lon-latlon_buf) & (df100_24h.lon <= this_lon+latlon_buf) & (df100_24h.lat >= this_lat-latlon_buf) & (df100_24h.lat <= this_lat+latlon_buf)] 
    more_rows_to_drop_10_24h = df10_24h[(df10_24h.time==this_time) & (df10_24h.lon >= this_lon-latlon_buf) & (df10_24h.lon <= this_lon+latlon_buf) & (df10_24h.lat >= this_lat-latlon_buf) & (df10_24h.lat <= this_lat+latlon_buf)] 
    
    rows_to_drop_100_24h = pd.concat([rows_to_drop_100_24h,more_rows_to_drop_100_24h])
    rows_to_drop_10_24h = pd.concat([rows_to_drop_10_24h,more_rows_to_drop_10_24h])  

#### drop duplicates
rows_to_drop_1000_72h = rows_to_drop_1000_72h.drop_duplicates()
rows_to_drop_100_72h.drop_duplicates(inplace=True)
rows_to_drop_10_72h.drop_duplicates(inplace=True)
rows_to_drop_1000_48h = rows_to_drop_1000_48h.drop_duplicates()
rows_to_drop_100_48h.drop_duplicates(inplace=True)
rows_to_drop_10_48h.drop_duplicates(inplace=True)
rows_to_drop_1000_24h = rows_to_drop_1000_24h.drop_duplicates()
rows_to_drop_100_24h.drop_duplicates(inplace=True)
rows_to_drop_10_24h.drop_duplicates(inplace=True)

rows_to_drop_100_72h


,time,lat,lon,tp,tp_minus_ari,tp_pct_of_ari,event_num


#### now drop those rows from the original dataframes. Method from here: https://www.includehelp.com/python/how-to-remove-rows-in-a-pandas-dataframe-if-the-same-row-exists-in-another-dataframe.aspx#:~:text=df1%20and%20df2.-,Remove%20rows%20in%20a%20Pandas%20dataframe%20if%20the%20same%20row,then%20drop%20the%20common%20rows.


In [195]:
df1000_72h_edit = pd.merge(df1000_72h,rows_to_drop_1000_72h,indicator=True,how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
df100_72h_edit = pd.merge(df100_72h,rows_to_drop_100_72h,indicator=True,how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
df10_72h_edit = pd.merge(df10_72h,rows_to_drop_10_72h,indicator=True,how='outer').query('_merge=="left_only"').drop('_merge', axis=1)

print("df1000_72h length went from "+str(len(df1000_72h))+" to "+str(len(df1000_72h_edit)))
print("df100_72h length went from "+str(len(df100_72h))+" to "+str(len(df100_72h_edit)))
print("df10_72h length went from "+str(len(df10_72h))+" to "+str(len(df10_72h_edit)))

#### 48h
df1000_48h_edit = pd.merge(df1000_48h,rows_to_drop_1000_48h, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
df100_48h_edit = pd.merge(df100_48h,rows_to_drop_100_48h, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
df10_48h_edit = pd.merge(df10_48h,rows_to_drop_10_48h, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)

print("df1000_48h length went from "+str(len(df1000_48h))+" to "+str(len(df1000_48h_edit)))
print("df100_48h length went from "+str(len(df100_48h))+" to "+str(len(df100_48h_edit)))
print("df10_48h length went from "+str(len(df10_48h))+" to "+str(len(df10_48h_edit)))

### 24h
df1000_24h_edit = pd.merge(df1000_24h,rows_to_drop_1000_24h, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
df100_24h_edit = pd.merge(df100_24h,rows_to_drop_100_24h, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
df10_24h_edit = pd.merge(df10_24h,rows_to_drop_10_24h, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)

print("df1000_24h length went from "+str(len(df1000_24h))+" to "+str(len(df1000_24h_edit)))
print("df100_24h length went from "+str(len(df100_24h))+" to "+str(len(df100_24h_edit)))
print("df10_24h length went from "+str(len(df10_24h))+" to "+str(len(df10_24h_edit)))

df1000_72h length went from 6 to 6
df100_72h length went from 216 to 216
df10_72h length went from 22384 to 22384
df1000_48h length went from 0 to 0
df100_48h length went from 97 to 97
df10_48h length went from 10659 to 10659
df1000_24h length went from 0 to 0
df100_24h length went from 40 to 40
df10_24h length went from 3532 to 3532


In [196]:
df1000_72h_edit.to_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_1000y72h_edit.csv", index=False)
df100_72h_edit.to_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_100y72h_edit.csv", index=False)
df10_72h_edit.to_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_10y72h_edit.csv", index=False)

rows_to_drop_1000_72h.to_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_1000y72h.csv", index=False)
rows_to_drop_100_72h.to_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_100y72h.csv", index=False)
rows_to_drop_10_72h.to_csv(workdir+"/"+dataset+"/72h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_10y72h.csv", index=False)
                             
#### 48h
df1000_48h_edit.to_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_1000y48h_edit.csv", index=False)
df100_48h_edit.to_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_100y48h_edit.csv", index=False)
df10_48h_edit.to_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_10y48h_edit.csv", index=False)

rows_to_drop_1000_48h.to_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_1000y48h.csv", index=False)
rows_to_drop_100_48h.to_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_100y48h.csv", index=False)
rows_to_drop_10_48h.to_csv(workdir+"/"+dataset+"/48h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_10y48h.csv", index=False)  

### 24h
df1000_24h_edit.to_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_1000y24h_edit.csv", index=False)
df100_24h_edit.to_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_100y24h_edit.csv", index=False)
df10_24h_edit.to_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_cleaned_10y24h_edit.csv", index=False)

rows_to_drop_1000_24h.to_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_1000y24h.csv", index=False)
rows_to_drop_100_24h.to_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_100y24h.csv", index=False)
rows_to_drop_10_24h.to_csv(workdir+"/"+dataset+"/24h/"+str(year)+"/"+dataset+"_"+str(year)+"_removed_points_10y24h.csv", index=False)
      